In [1]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv("data.csv")

In [5]:
df.head()

,url,status
0,http://www.crestonwood.com/router.php,legitimate
1,http://shadetreetechnology.com/V4/validation/a...,phishing
2,https://support-appleld.com.secureupdate.duila...,phishing
3,http://rgipt.ac.in,legitimate
4,http://www.iracing.com/tracks/gateway-motorspo...,legitimate


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11430 entries, 0 to 11429
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   url     11430 non-null  object
 1   status  11430 non-null  object
dtypes: object(2)
memory usage: 178.7+ KB


In [21]:
df["status"].unique()

array(['legitimate', 'phishing'], dtype=object)

In [9]:
df.isnull().sum()

url       0
status    0
dtype: int64

In [11]:
import re

def extract_features(url):
    return [
        len(url),
        url.count('.'),
        sum(1 for c in url if c.isdigit()),
        sum(1 for c in url if c in ['@','?','-','=']),
        int('https' in url),
        int('@' in url),
        int(bool(re.search(r'\d+\.\d+\.\d+\.\d+', url))),
        int(any(x in url for x in ['bit.ly','tinyurl','goo.gl']))
    ]


In [13]:
X = df['url'].apply(extract_features).tolist()
X = np.array(X)

y = df['status']


In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [17]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)


LogisticRegression(max_iter=1000)

In [19]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.6907261592300963
              precision    recall  f1-score   support

  legitimate       0.66      0.81      0.73      1157
    phishing       0.74      0.57      0.65      1129

    accuracy                           0.69      2286
   macro avg       0.70      0.69      0.69      2286
weighted avg       0.70      0.69      0.69      2286



In [23]:
test_url = "http://amozon.in/login"

features = np.array(extract_features(test_url)).reshape(1, -1)
prediction = model.predict(features)

print("Phishing" if prediction[0]==1 else "Legitimate")


Legitimate


In [27]:
def check_url(url):
    f = np.array(extract_features(url)).reshape(1, -1)
    return "Phishing Link" if model.predict(f)[0]==1 else "Safe Link"


In [29]:
check_url("goolge.com")

'Safe Link'